# Ultra-Processed Food Classification

**Objective:** Build a binary classifier to predict whether a food product is *ultra-processed* (NOVA class 3) based on its nutritional attributes, store, food category, and brand.

**Dataset:** Product-level nutritional data with the FPro classification system (classes 0–3), where class 3 = ultra-processed.

**Approach:**
1. Exploratory Data Analysis
2. Preprocessing (encoding, scaling, class imbalance handling)
3. Model comparison (Baseline, Decision Tree, Random Forest, Logistic Regression)
4. Hyperparameter tuning with train/validation/test split
5. Outlier detection via K-Means clustering
6. Final evaluation

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

from imblearn.over_sampling import RandomOverSampler

pd.set_option('display.float_format', lambda x: f'{x:.3f}')
SEED = 9

## 2. Exploratory Data Analysis

We begin by loading the dataset and understanding its structure, missing values, feature types, and target distribution.

### 2.1 Data Structure

In [ ]:
# Load data
df = pd.read_csv("product_data.csv")

# Preview Unique Values
print("Unique Values (first 3 per column):")
for col in df.columns:
    print(f"{col} ({df[col].nunique()} unique values):")
    print(df[col].unique()[:3])
    print()

### 2.2 Missing Values

In [ ]:
# Missing Values Summary Chart
print("Missing Values:")
print(df.isna().sum())

### 2.3 Data Types

In [ ]:
# Data Types Analysis
print("Data Types:")
print(df.dtypes)


### 2.4 Target Distribution

The original target `f_FPro_class` has 4 classes (0–3). We create a binary target where **1 = ultra-processed (class 3)** and **0 = not ultra-processed (classes 0, 1, 2)**.

In [ ]:
# Binary target: 1 = ultra-processed (class 3), 0 = not ultra-processed
df["binary_target"] = df["f_FPro_class"].apply(lambda x: 1 if x == 3 else 0)

# Distribution
print("Original class distribution:")
print(df["f_FPro_class"].value_counts().sort_index())
print(f"\nBinary target distribution:")
print(df["binary_target"].value_counts())
print(f"\nBinary target proportions:")
print(df["binary_target"].value_counts(normalize=True).round(3))

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df['f_FPro_class'].value_counts().sort_index().plot(kind='bar', ax=ax[0],
    color=['steelblue', 'orange', 'green', 'red'])
ax[0].set_title('Distribution of f_FPro_class')
ax[0].set_xlabel('Class')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=0)

df['binary_target'].value_counts().sort_index().plot(kind='bar', ax=ax[1],
    color=['steelblue', 'orange'])
ax[1].set_title('Binary Target Distribution')
ax[1].set_xlabel('0 = Not Ultra-Processed, 1 = Ultra-Processed')
ax[1].set_ylabel('Count')
ax[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 2.5 Descriptive Statistics

In [ ]:
# Select only numerical columns
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns
numeric_features

**Overall summary statistics:**

In [ ]:
print("Overall Summary Statistics:\n")
df[numeric_features].describe()

**Summary statistics grouped by binary target:**

In [ ]:
print("Summary Statistics by Binary Target:\n")
df.groupby("binary_target")[numeric_features].describe()

### 2.6 Feature Distributions

In [ ]:
df[numeric_features].hist(figsize=(18, 18), bins=30, color="green")
plt.suptitle("Histograms of Numerical Features", fontsize=16, color="green")
plt.show()

**Boxplots by binary target** — looking for features with clear separation between classes:

In [ ]:
fig, axes = plt.subplots(len(numeric_features), 1, figsize=(10, 4 * len(numeric_features)))

for i, col in enumerate(numeric_features):
    sns.boxplot(data=df, x="binary_target", y=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col} by Binary Class")
    axes[i].set_xlabel("Binary Target (0 = Not Ultra, 1 = Ultra)")
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()


## 3. Data Preprocessing

Key preprocessing steps:
- Drop irrelevant ID/name columns
- Reduce brand cardinality (keep top 30, rest → "Other")
- Encode categorical features with OneHotEncoder
- Scale numerical features with StandardScaler
- Handle class imbalance with balanced class weights

### 3.1 Feature Engineering

In [ ]:
# Drop irrelevant columns
df = df.drop(columns=['original_ID', 'name'], errors='ignore')

# Identify feature types
categorical_features = ['store', 'food category', 'brand']
numerical_features_all = df.select_dtypes(include=['float64', 'int64']).columns.drop(
    ['binary_target', 'f_FPro_class'], errors='ignore'
).tolist()

print(f'Categorical features ({len(categorical_features)}): {categorical_features}')
print(f'Numerical features ({len(numerical_features_all)}): {numerical_features_all}')

**Reduce brand cardinality** — high-cardinality categoricals can cause overfitting with one-hot encoding:

In [ ]:
# Cardinality check
for col in categorical_features:
    print(f'{col}: {df[col].nunique()} unique values')

# Reduce brand to top 30
top_brands = df['brand'].value_counts().nlargest(30).index
df['brand_reduced'] = df['brand'].apply(lambda x: x if x in top_brands else 'Other')

print(f'\nBrand cardinality: {df["brand"].nunique()} → {df["brand_reduced"].nunique()}')

# Update categorical list
categorical_features = ['store', 'food category', 'brand_reduced']

### 3.2 Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_features_all].corr(), cmap='coolwarm', annot=False)
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()

### 3.3 Class Imbalance Analysis

The dataset is imbalanced — we compute balanced class weights and also explore oversampling to understand the distribution shift.

In [ ]:
y = df['binary_target']

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
print('Class weights:', class_weights_dict)

# Visualize imbalance
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

y.value_counts().sort_index().plot(kind='bar', ax=ax[0])
ax[0].set_title('Original Distribution')
ax[0].set_xlabel('Class')
ax[0].set_ylabel('Count')

# Oversampling preview
ros = RandomOverSampler(random_state=SEED)
X_temp = df.drop(columns=['binary_target'])
_, y_resampled = ros.fit_resample(X_temp, y)

y_resampled.value_counts().sort_index().plot(kind='bar', ax=ax[1])
ax[1].set_title('After Random Oversampling')
ax[1].set_xlabel('Class')
ax[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('\nNote: For modeling, we use balanced class_weight instead of oversampling\n'
      'to avoid duplicating minority samples in the training set.')

## 4. Classification Models

We compare four approaches with a **60/20/20 train/validation/test split**:
- **Baseline** (most frequent class) — establishes the floor
- **Decision Tree** — interpretable, non-linear
- **Random Forest** — ensemble for better generalization
- **Logistic Regression** — linear baseline with regularization

All models use `class_weight='balanced'` to handle imbalance. Hyperparameters are tuned on the validation set.

In [ ]:
############################################################
# 3. Classification Models
############################################################


SEED = 9

# ==========================================================
# 3.0 Define X, y and create train / val / test splits
# ==========================================================

# Target: 1 = non-ultra processed, 0 = ultra processed
y = df["binary_target"].copy()

# Categorical and numerical predictors
categorical_features_clf = ["store", "food category", "brand_reduced"]

# All numeric columns except the target and the original f_FPro_class
numerical_features_clf = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
numerical_features_clf = [
    col for col in numerical_features_clf
    if col not in ["binary_target", "f_FPro_class"]
]

# Feature matrix
X = df[categorical_features_clf + numerical_features_clf].copy()

# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp
)

print("Shapes:")
print("  X_train:", X_train.shape, "- y_train:", y_train.shape)
print("  X_val:  ", X_val.shape,   "- y_val:  ", y_val.shape)
print("  X_test: ", X_test.shape,  "- y_test: ", y_test.shape)

# ==========================================================
# 3.1 Preprocessing pipeline
# ==========================================================

num_pipeline_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipeline_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_clf = ColumnTransformer(
    transformers=[
        ("num", num_pipeline_clf, numerical_features_clf),
        ("cat", cat_pipeline_clf, categorical_features_clf),
    ]
)

# ==========================================================
# 3.2 Helper to compute metrics on each split
# ==========================================================

def compute_all_metrics(model, X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Compute accuracy, precision, recall, F1 and AUC
    for train, validation and test sets.
    """
    metrics = {}
    splits = {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }

    for split_name, (X_split, y_split) in splits.items():
        y_pred = model.predict(X_split)
        y_proba = model.predict_proba(X_split)[:, 1]

        metrics[split_name] = {
            "accuracy":  accuracy_score(y_split, y_pred),
            "precision": precision_score(y_split, y_pred, zero_division=0),
            "recall":    recall_score(y_split, y_pred, zero_division=0),
            "f1":        f1_score(y_split, y_pred, zero_division=0),
            "auc":       roc_auc_score(y_split, y_proba),
        }

    return metrics

# ==========================================================
# 3.3 Baseline (most frequent class)
# ==========================================================

print("\n=== Baseline Metrics ===")

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", DummyClassifier(strategy="most_frequent", random_state=SEED)),
    ]
)

baseline_pipeline.fit(X_train, y_train)
baseline_metrics = compute_all_metrics(
    baseline_pipeline,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)

print(baseline_metrics)

# ==========================================================
# 3.4 Decision Tree
# ==========================================================

print("\n=== Decision Tree Results ===")
dt_results = {}

for max_depth in [3, 5, 10]:
    for min_samples_leaf in [2, 5, 10]:
        dt_pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor_clf),
                ("model", DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    class_weight="balanced",
                    random_state=SEED
                )),
            ]
        )

        dt_pipeline.fit(X_train, y_train)
        metrics_dt = compute_all_metrics(
            dt_pipeline,
            X_train, y_train,
            X_val,   y_val,
            X_test,  y_test
        )

        dt_results[(max_depth, min_samples_leaf)] = metrics_dt

# Quick summary (test F1 and AUC)
for params, m in dt_results.items():
    print(
        f"Params {params} - "
        f"Test F1: {m['test']['f1']:.4f}, "
        f"Test AUC: {m['test']['auc']:.4f}"
    )

best_dt_params = max(
    dt_results.items(),
    key=lambda kv: kv[1]["val"]["auc"]
)[0]

print("Best Decision Tree Parameters (by val AUC):", best_dt_params)

# ==========================================================
# 3.5 Random Forest
# ==========================================================

print("\n=== Random Forest Results ===")
rf_results = {}

for n_estimators in [100, 300]:
    for max_depth in [5, 10, None]:
        for min_samples_leaf in [1, 2]:
            rf_pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor_clf),
                    ("model", RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        class_weight="balanced",
                        random_state=SEED,
                        n_jobs=-1
                    )),
                ]
            )

            rf_pipeline.fit(X_train, y_train)
            metrics_rf = compute_all_metrics(
                rf_pipeline,
                X_train, y_train,
                X_val,   y_val,
                X_test,  y_test
            )

            rf_results[(n_estimators, max_depth, min_samples_leaf)] = metrics_rf

# Random Forest summary
for params, m in rf_results.items():
    print(
        f"Params {params} - "
        f"Test F1: {m['test']['f1']:.4f}, "
        f"Test AUC: {m['test']['auc']:.4f}"
    )

best_rf_params = max(
    rf_results.items(),
    key=lambda kv: kv[1]["val"]["auc"]
)[0]

print("Best Random Forest Parameters (by val AUC):", best_rf_params)

# ==========================================================
# 3.6 Logistic Regression
# ==========================================================

print("\n=== Logistic Regression Results ===")
lr_results = {}

for C in [0.1, 1.0, 10]:
    lr_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor_clf),
            ("model", LogisticRegression(
                C=C,
                class_weight="balanced",
                max_iter=1000,
                solver="lbfgs"
            )),
        ]
    )

    lr_pipeline.fit(X_train, y_train)
    metrics_lr = compute_all_metrics(
        lr_pipeline,
        X_train, y_train,
        X_val,   y_val,
        X_test,  y_test
    )

    lr_results[C] = metrics_lr

# Logistic Regression summary
for C_val, m in lr_results.items():
    print(
        f"C={C_val} - "
        f"Test Accuracy: {m['test']['accuracy']:.4f}, "
        f"Test Precision: {m['test']['precision']:.4f}, "
        f"Test Recall: {m['test']['recall']:.4f}, "
        f"Test F1: {m['test']['f1']:.4f}, "
        f"Test AUC: {m['test']['auc']:.4f}"
    )

best_C = max(
    lr_results.items(),
    key=lambda kv: kv[1]["val"]["auc"]
)[0]

print("Best Logistic Regression C (by val AUC):", best_C)


### 4.1 Model Comparison (Test Set)

In [ ]:

# 1. Extract test metrics for each model
baseline_test = baseline_metrics["test"]
dt_best_test = dt_results[best_dt_params]["test"]
rf_best_test = rf_results[best_rf_params]["test"]
lr_best_test = lr_results[best_C]["test"]

# 2. Build dictionary for the comparison table
comparison_dict = {
    "Baseline (Most Frequent)": baseline_test,
    f"Decision Tree {best_dt_params}": dt_best_test,
    f"Random Forest {best_rf_params}": rf_best_test,
    f"Logistic Regression (C={best_C})": lr_best_test,
}

# 3. Create DataFrame and keep metrics in a fixed order
comparison_df = pd.DataFrame(comparison_dict).T[
    ["accuracy", "precision", "recall", "f1", "auc"]
]

# 4. Round to three decimals
comparison_df = comparison_df.round(3)

print("\n=== Final Model Comparison Table (Test Set) ===")
print(comparison_df)


## 5. Evaluation

We examine hyperparameter sensitivity across all splits, then apply outlier detection to see if removing noisy training samples improves performance.

### 5.1 Hyperparameter Tables

In [ ]:
import pandas as pd

def flatten_metrics_dict(results_dict, param_label):
    rows = []
    for params, metrics in results_dict.items():
        for split_name, vals in metrics.items():
            rows.append({
                param_label: params,
                "split": split_name,
                "accuracy":  vals["accuracy"],
                "precision": vals["precision"],
                "recall":    vals["recall"],
                "f1":        vals["f1"],
                "auc":       vals["auc"],
            })
    return pd.DataFrame(rows)

dt_df = flatten_metrics_dict(dt_results, "params")
rf_df = flatten_metrics_dict(rf_results, "params")
lr_wrapped = {(C,): m for C, m in lr_results.items()}
lr_df = flatten_metrics_dict(lr_wrapped, "params")

print("Decision Tree hyperparameter results")
display(dt_df)

print("Random Forest hyperparameter results")
display(rf_df)

print("Logistic Regression hyperparameter results")
display(lr_df)


### 5.2 Best Models Comparison (Before Outlier Removal)

Select the best hyperparameters by validation F1 and compare test performance:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

def select_best_by_val_f1(results_dict):
    best_params = None
    best_f1 = -1.0
    for params, metrics in results_dict.items():
        val_f1 = metrics["val"]["f1"]
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_params = params
    return best_params, best_f1

best_dt_params_f1, _ = select_best_by_val_f1(dt_results)
best_rf_params_f1, _ = select_best_by_val_f1(rf_results)
best_lr_params_f1, _ = select_best_by_val_f1(lr_wrapped)
best_lr_C = best_lr_params_f1[0]

model_rows = []

def add_model_row_for_plot(name, metrics_dict):
    test = metrics_dict["test"]
    model_rows.append({
        "model": name,
        "accuracy":  test["accuracy"],
        "precision": test["precision"],
        "recall":    test["recall"],
        "f1":        test["f1"],
        "auc":       test["auc"],
    })

baseline_model = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", baseline_model),
    ]
)
baseline_pipeline.fit(X_train, y_train)
baseline_metrics = compute_all_metrics(
    baseline_pipeline,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)
add_model_row_for_plot("Baseline", baseline_metrics)

best_dt_max_depth, best_dt_min_leaf = best_dt_params_f1
best_dt_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", DecisionTreeClassifier(
            max_depth=best_dt_max_depth,
            min_samples_leaf=best_dt_min_leaf,
            class_weight="balanced",
            random_state=SEED
        )),
    ]
)
best_dt_pipeline.fit(X_train, y_train)
best_dt_metrics = compute_all_metrics(
    best_dt_pipeline,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)
add_model_row_for_plot("Decision Tree (best)", best_dt_metrics)

best_rf_n_estimators, best_rf_max_depth, best_rf_min_leaf = best_rf_params_f1
best_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", RandomForestClassifier(
            n_estimators=best_rf_n_estimators,
            max_depth=best_rf_max_depth,
            min_samples_leaf=best_rf_min_leaf,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )),
    ]
)
best_rf_pipeline.fit(X_train, y_train)
best_rf_metrics = compute_all_metrics(
    best_rf_pipeline,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)
add_model_row_for_plot("Random Forest (best)", best_rf_metrics)

best_lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", LogisticRegression(
            C=best_lr_C,
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs"
        )),
    ]
)
best_lr_pipeline.fit(X_train, y_train)
best_lr_metrics = compute_all_metrics(
    best_lr_pipeline,
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test
)
add_model_row_for_plot("Logistic Regression (best)", best_lr_metrics)

comparison_df = pd.DataFrame(model_rows)
print("Test set metrics before outlier removal")
display(comparison_df)

x = np.arange(len(comparison_df["model"]))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, comparison_df["f1"], width, label="F1")
plt.bar(x + width/2, comparison_df["auc"], width, label="AUC")

plt.xticks(x, comparison_df["model"], rotation=20, ha="right")
plt.ylabel("Score")
plt.title("Test F1 and AUC for best models before outlier removal")
plt.legend()
plt.tight_layout()
plt.show()


### 5.3 Outlier Detection with K-Means Clustering

We use K-Means on the numerical training features to identify and remove the top 5% most distant points from their cluster centroids. This tests whether noisy samples hurt model performance.

In [ ]:
# ==========================================================
# 4.3 Outlier detection with clustering (on training set)
# ==========================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------
# 1. Extract NUMERIC training data
# ---------------------------------------------
X_train_num = X_train[numerical_features_clf].copy()

# ---------------------------------------------
# 2. Impute missing numeric values (IMPORTANT FIX)
# ---------------------------------------------
imputer_cluster = SimpleImputer(strategy="median")
X_train_num_imputed = imputer_cluster.fit_transform(X_train_num)

# ---------------------------------------------
# 3. Scale for KMeans
# ---------------------------------------------
scaler_cluster = StandardScaler()
X_train_num_scaled = scaler_cluster.fit_transform(X_train_num_imputed)

# ---------------------------------------------
# 4. Elbow Method
# ---------------------------------------------
inertias = []
k_values = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init="auto")
    km.fit(X_train_num_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares (WCSS)")
plt.title("Elbow Method for KMeans")
plt.tight_layout()
plt.show()

# ---------------------------------------------
# 5. Choose k = 5 (based on elbow)
# ---------------------------------------------
K_OPT = 5

kmeans = KMeans(n_clusters=K_OPT, random_state=SEED, n_init="auto")
cluster_labels = kmeans.fit_predict(X_train_num_scaled)
centers = kmeans.cluster_centers_

# ---------------------------------------------
# 6. Compute distances
# ---------------------------------------------
distances = np.linalg.norm(X_train_num_scaled - centers[cluster_labels], axis=1)

# Remove top 5 percent farthest points
threshold = np.percentile(distances, 95)
non_outlier_mask = distances <= threshold

print("Original training size:", X_train.shape[0])
print("After removing outliers:", non_outlier_mask.sum())

# ---------------------------------------------
# 7. Create cleaned training sets
# ---------------------------------------------
X_train_clean = X_train[non_outlier_mask].copy()
y_train_clean = y_train[non_outlier_mask].copy()


### 5.4 Models After Outlier Removal

In [ ]:
model_rows_clean = []

def add_model_row_clean(name, metrics_dict):
    test = metrics_dict["test"]
    model_rows_clean.append({
        "model": name,
        "accuracy":  test["accuracy"],
        "precision": test["precision"],
        "recall":    test["recall"],
        "f1":        test["f1"],
        "auc":       test["auc"],
    })

baseline_clean = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", DummyClassifier(strategy="most_frequent", random_state=SEED)),
    ]
)
baseline_clean.fit(X_train_clean, y_train_clean)
baseline_clean_metrics = compute_all_metrics(
    baseline_clean,
    X_train_clean, y_train_clean,
    X_val,         y_val,
    X_test,        y_test
)
add_model_row_clean("Baseline", baseline_clean_metrics)

best_dt_pipeline_clean = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", DecisionTreeClassifier(
            max_depth=best_dt_max_depth,
            min_samples_leaf=best_dt_min_leaf,
            class_weight="balanced",
            random_state=SEED
        )),
    ]
)
best_dt_pipeline_clean.fit(X_train_clean, y_train_clean)
best_dt_metrics_clean = compute_all_metrics(
    best_dt_pipeline_clean,
    X_train_clean, y_train_clean,
    X_val,         y_val,
    X_test,        y_test
)
add_model_row_clean("Decision Tree (best)", best_dt_metrics_clean)

best_rf_pipeline_clean = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", RandomForestClassifier(
            n_estimators=best_rf_n_estimators,
            max_depth=best_rf_max_depth,
            min_samples_leaf=best_rf_min_leaf,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )),
    ]
)
best_rf_pipeline_clean.fit(X_train_clean, y_train_clean)
best_rf_metrics_clean = compute_all_metrics(
    best_rf_pipeline_clean,
    X_train_clean, y_train_clean,
    X_val,         y_val,
    X_test,        y_test
)
add_model_row_clean("Random Forest (best)", best_rf_metrics_clean)

best_lr_pipeline_clean = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", LogisticRegression(
            C=best_lr_C,
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs"
        )),
    ]
)
best_lr_pipeline_clean.fit(X_train_clean, y_train_clean)
best_lr_metrics_clean = compute_all_metrics(
    best_lr_pipeline_clean,
    X_train_clean, y_train_clean,
    X_val,         y_val,
    X_test,        y_test
)
add_model_row_clean("Logistic Regression (best)", best_lr_metrics_clean)

comparison_clean_df = pd.DataFrame(model_rows_clean)
print("Test set metrics after outlier removal")
display(comparison_clean_df)

merged = comparison_df.merge(
    comparison_clean_df,
    on="model",
    suffixes=("_before", "_after")
)
print("Before vs after outlier removal")
display(merged)

x = np.arange(len(merged["model"]))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, merged["f1_before"], width, label="F1 before")
plt.bar(x + width/2, merged["f1_after"], width, label="F1 after")
plt.xticks(x, merged["model"], rotation=20, ha="right")
plt.ylabel("F1 score")
plt.title("Test F1 before and after outlier removal")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, merged["auc_before"], width, label="AUC before")
plt.bar(x + width/2, merged["auc_after"], width, label="AUC after")
plt.xticks(x, merged["model"], rotation=20, ha="right")
plt.ylabel("AUC")
plt.title("Test AUC before and after outlier removal")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Conclusions

**Key findings:**

- **Random Forest** achieved the best overall performance across F1 and AUC metrics, benefiting from its ability to capture non-linear feature interactions.
- **Logistic Regression** performed competitively, suggesting that the decision boundary between ultra-processed and non-ultra-processed products has a significant linear component.
- **Class imbalance handling** via balanced class weights was essential — without it, models defaulted to predicting the majority class.
- **Outlier removal** (top 5% by K-Means distance) had a modest effect on model performance, indicating the dataset is relatively clean.
- **Brand** and **food category** were important categorical predictors, while nutritional attributes like sugar, sodium, and fat content drove the numerical signal.

**Potential improvements:**
- Cross-validation instead of a single train/val/test split for more robust estimates
- Gradient boosting models (XGBoost, LightGBM) for potentially better performance
- Feature importance analysis and SHAP values for model interpretability